# Un livre Shamela → un fichier SQLite

Ce notebook transforme **un seul livre** du corpus Shamela 4 en un fichier `book.sqlite`
lisible par l'application, en s'arrêtant à chaque étape pour regarder les données.

C'est la version pédagogique de `tools/import_shamela.py`, qui fait la même chose
sur 8 589 livres en parallèle. Ici on privilégie la lisibilité sur la performance.

## Rappel de l'architecture

L'application est *local-first* : pas d'API, trois bases SQLite séparées.

| Base | Contenu | Accès |
|---|---|---|
| `catalog.sqlite` | catalogue de **tous** les livres (titres, auteurs, catégories) | lecture seule |
| `books/<edition_id>.sqlite` | le contenu **d'un** livre (pages, volumes, sommaire) | lecture seule |
| `user.sqlite` | bibliothèque, progression, réglages | lecture/écriture |

On ne fabrique ici que le **deuxième** : un fichier livre. Le catalogue est
construit à la fin d'un import complet, une fois tous les livres connus.

### Pourquoi un fichier par livre ?

Parce qu'un livre se télécharge, se supprime, se met à jour et se vérifie
indépendamment des autres. Une base unique de 19 Go ne permettrait rien de tout ça.

## Le livre de démonstration

`دراسات في الديانات الهندية` (id 5925, catégorie العقيدة) : 35 pages, 5 volumes,
des notes de bas de page, des titres. Assez petit pour tout inspecter à l'œil,
assez riche pour rencontrer les vrais pièges.

In [ ]:
import io
import json
import os
import re
import sqlite3
import sys

# La console Windows est en cp1252 : sans ça, le premier titre arabe imprimé
# fait planter le kernel avec UnicodeEncodeError. Réflexe obligatoire.
sys.stdout.reconfigure(encoding="utf-8", errors="replace")

# Racine du dépôt (le notebook vit dans tools/notebooks/)
REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if os.path.basename(REPO) != "beytelhikma":
    REPO = r"C:\dev\horizon\beytelhikma\beytelhikma"

# tools/_common.py contient le DDL et la normalisation, partagés avec le
# générateur d'exemple ET l'importeur. On importe, on ne recopie jamais :
# c'est ce qui rend une dérive de schéma impossible.
sys.path.insert(0, os.path.join(REPO, "tools"))
from _common import (  # noqa: E402
    BOOK_SCHEMA,
    SCHEMA_VERSION,
    decode_entities,
    normalize_ar,
    sha256_file,
    sha256_text,
    strip_html,
)

SHAMELA = r"C:\shamela-data"
BOOK_DIR = os.path.join(SHAMELA, "01__العقيدة", "5925__دراسات-في-الديانات-الهندية")

OUT_DIR = os.path.join(REPO, "dist", "demo")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_DB = os.path.join(OUT_DIR, "sh-5925.sqlite")

print("dépôt   :", REPO)
print("source  :", BOOK_DIR)
print("sortie  :", OUT_DB)
print("source existe :", os.path.isdir(BOOK_DIR))

## Étape 0 — Les quatre fichiers source

Chaque livre du corpus est un dossier de quatre fichiers, toujours les mêmes
(vérifié sur les 8 589 livres : aucun manquant, aucun fichier en trop).

| Fichier | Rôle |
|---|---|
| `manifest.json` | compteurs + **empreintes SHA-256** des trois autres fichiers |
| `book_metadata.json` | titre, auteurs, catégorie, notice bibliographique |
| `toc.jsonl` | sommaire, une entrée par ligne |
| `pages.jsonl` | contenu, une page par ligne |

`.jsonl` = *JSON Lines* : un objet JSON complet **par ligne**. C'est ce qui permet
de lire un fichier de 251 Mo sans jamais le charger entièrement en mémoire.

Le `manifest.json` est notre filet de sécurité : il donne le nombre de lignes
attendu et le SHA-256 de chaque fichier. On s'en servira pour détecter une
source corrompue **avant** d'écrire quoi que ce soit.

In [ ]:
for name in sorted(os.listdir(BOOK_DIR)):
    size = os.path.getsize(os.path.join(BOOK_DIR, name))
    print(f"{name:22s} {size:>10,} octets")

manifest = json.load(io.open(os.path.join(BOOK_DIR, "manifest.json"), encoding="utf-8"))
print("\n--- manifest.json ---")
print(json.dumps(manifest, ensure_ascii=False, indent=2)[:900])

## Étape 1 — Les métadonnées, et le champ `betaka_text`

`book_metadata.json` donne le titre, la catégorie et les auteurs sous forme
structurée. Mais l'information la plus riche — éditeur, numéro d'édition, année,
muhaqqiq — n'est **pas** structurée : elle est enfouie dans `betaka_text`, un bloc
de texte libre où les lignes sont séparées par des `\r` (retour chariot seul,
pas `\n`).

On en extrait ce qu'on peut par préfixe (`الناشر:` = éditeur, `الطبعة:` = édition).
Ces préfixes sont présents dans 85 % et 69 % des livres du corpus.

**Deux pièges de dates :**

1. `death_hijri = 99999` n'est pas une année, c'est une sentinelle « inconnu »
   (1 069 auteurs sur 3 187). Elle doit devenir `NULL`, sinon l'app affichera
   « mort en l'an 99999 ».
2. Les années dans `betaka_text` sont majoritairement hégiriennes (`١٤٣٩ هـ`).
   La colonne `publication_year` de l'app est grégorienne : on n'y met une valeur
   que si on trouve explicitement un millésime suivi de `م`.

In [ ]:
meta = json.load(io.open(os.path.join(BOOK_DIR, "book_metadata.json"), encoding="utf-8"))

for key in ("book_id", "shamela_id", "title_ar", "category_id", "category_name_ar",
            "main_author_id", "main_author_name_ar", "main_author_death_hijri",
            "volume_count_observed", "has_multi_part", "is_hidden"):
    print(f"{key:26s} = {meta[key]!r}")

print("\nauteurs :", json.dumps(meta["authors"], ensure_ascii=False))

# betaka_text : lignes séparées par \r, invisibles si on imprime tel quel.
print("\n--- betaka_text, ligne par ligne ---")
for line in meta["betaka_text"].split("\r"):
    if line.strip():
        print(" ", line.strip())

In [ ]:
# --- extraction des champs bibliographiques -------------------------------

ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")


def betaka_field(betaka: str, label: str) -> str | None:
    """Valeur de la ligne `<label>: ...` dans betaka_text, ou None."""
    for line in betaka.split("\r"):
        line = line.strip()
        if line.startswith(label):
            value = line[len(label):].lstrip(": \u061b").strip()
            return value or None
    return None


def gregorian_year(betaka: str) -> int | None:
    """Année grégorienne : un millésime suivi de `م`, jamais une année hégirienne."""
    haystack = " ".join(
        v for v in (betaka_field(betaka, "الطبعة"), betaka_field(betaka, "عام النشر")) if v
    ).translate(ARABIC_DIGITS)
    m = re.search(r"\b(1[5-9]\d{2}|20\d{2})\s*م", haystack)
    return int(m.group(1)) if m else None


def clean_death(value) -> int | None:
    """99999 = sentinelle « date inconnue » côté Shamela."""
    return None if value in (None, 0, 99999) or value < 0 else int(value)


betaka = meta["betaka_text"]
print("éditeur         :", betaka_field(betaka, "الناشر"))
print("édition         :", betaka_field(betaka, "الطبعة"))
print("muhaqqiq        :", betaka_field(betaka, "المحقق"))
print("année grégo.    :", gregorian_year(betaka))
print("décès (brut)    :", meta["main_author_death_hijri"])
print("décès (nettoyé) :", clean_death(meta["main_author_death_hijri"]))

## Étape 2 — Les pages brutes

Chaque ligne de `pages.jsonl` a dix champs. Trois d'entre eux réservent des surprises,
mesurées sur les 7,6 millions de pages du corpus :

| Champ | Réalité mesurée |
|---|---|
| `part` | une **chaîne**, pas un entier (`"6"`, parfois `"مقدمة"`). Non-null sur 82,5 % des pages |
| `hints` | **null sur les 7 611 186 pages**. Champ mort, on le mappe sur `NULL` sans le lire |
| `services_raw` | une chaîne contenant du JSON (pas un objet), présente sur 24 livres seulement |
| `sequence_num` | **redémarre à 1 à chaque volume** et comporte des doublons dans 25 livres sur 150 échantillonnés — inutilisable comme ordre |

Et le `body` n'est pas du texte plat : les paragraphes y sont séparés par des `\r`,
et il contient un HTML minimal (`<span data-type='title'>`, `<hr>`, parfois des tables
et des images en base64).

In [ ]:
def read_jsonl(path):
    """Lecture ligne à ligne. `newline=''` : on ne veut aucune traduction de fin
    de ligne, les `\\r` sont des données, pas de la mise en forme."""
    with io.open(path, encoding="utf-8", newline="") as fh:
        for line in fh:
            if line.strip():
                yield json.loads(line)


pages = list(read_jsonl(os.path.join(BOOK_DIR, "pages.jsonl")))
print("pages lues :", len(pages), "| annoncé par le manifest :", manifest["page_count"])

print("\nchamps d'une page :", list(pages[0].keys()))

print("\npage_id    part  page_num  seq  taille  notes")
for p in pages[:10]:
    print(f"{p['page_id']:<10} {str(p['part']):<5} {str(p['page_num']):<9} "
          f"{p['sequence_num']:<4} {len(p['body']):<7} {p['footnotes'] is not None}")

print("\nhints tous nuls        :", all(p["hints"] is None for p in pages))
print("services_raw tous nuls :", all(p["services_raw"] is None for p in pages))

In [ ]:
# Le body vu tel qu'il est stocké : les \r sont visibles en repr()
sample = pages[1]["body"]
print(repr(sample[:400]))

print("\nnombre de segments séparés par \\r :", sample.count("\r") + 1)
print("contient un \\n :", "\n" in sample, " <- toujours False dans ce corpus")

# Une page qui porte un titre
titled = next(p for p in pages if "data-type" in p["body"])
print("\n--- page avec titre (page_id", titled["page_id"], ") ---")
print(repr(titled["body"][:300]))

## Étape 3 — L'ordre de lecture et les volumes

**C'est le piège principal du corpus.**

On pourrait croire que l'ordre de lecture est `(part, sequence_num)`. C'est faux :
sur 150 livres testés, ce tri produit 2 417 inversions de pagination, contre 276
pour un simple tri par `page_id`. Et `sequence_num` a des doublons.

**Règle retenue : l'ordre de lecture est `page_id` croissant.** C'est une clé
primaire côté base source, donc un ordre stable et global.

Une fois les pages triées par `page_id`, les `part` forment des blocs contigus
(vérifié sur 150/150 livres) : chaque changement de `part` ouvre un volume.

### Le cas `part_number`

La colonne `volumes.part_number` est `INTEGER NOT NULL`. Or `part` est une chaîne
qui peut valoir `"مقدمة"` (« introduction ») dans environ 7 % des livres.

La règle naïve — numéro imprimé si numérique, position sinon — **crée une
collision** : un livre dont les parts sont `["مقدمة", "1"]` donnerait `part_number = 1`
aux deux volumes (le premier par repli sur la position, le second par son numéro
imprimé), et l'interface afficherait deux « tome 1 ».

On tranche donc pour le livre entier, la cohérence interne primant sur la fidélité
volume par volume :

- **toutes** les parts numériques → `part_number = int(part)`
- une seule ne l'est pas (ou aucune part) → `part_number = position`, et la valeur
  réelle survit dans `label_ar`

Ce livre illustre l'intérêt du premier cas : ses volumes sont numérotés **6 à 10**
(ce sont des numéros de livraison d'une revue). Le numéro imprimé et la position
dans le livre sont deux choses différentes.

In [ ]:
# 1. Ordre de lecture
pages.sort(key=lambda p: p["page_id"])

# 2. `part_number` doit rester unique dans le livre, sinon l'interface affiche
#    deux « tome 1 ». Un livre mêlant "مقدمة" et "1" produirait exactement cette
#    collision si on décidait volume par volume. On tranche donc pour tout le
#    livre : numéros imprimés seulement s'ils le sont tous.
raw_parts = {(p["part"] or "").strip() for p in pages}
use_printed = all(raw.isdigit() for raw in raw_parts if raw) and any(raw_parts - {""})
print("parts brutes :", sorted(raw_parts), "| numéros imprimés utilisables :", use_printed)

# 3. Découpage en volumes : chaque changement de `part` ouvre un volume
volumes = []          # lignes de la table `volumes`
volume_of_page = {}   # page_id -> volume_id
previous_part = object()  # sentinelle : ne peut égaler aucune valeur réelle

for page in pages:
    part = page["part"]
    if part != previous_part:
        ordinal = len(volumes) + 1
        raw = (part or "").strip()
        numeric = raw.isdigit()
        volumes.append({
            "volume_id": ordinal,
            "sequence_num": ordinal,
            "part_number": int(raw) if use_printed else ordinal,
            # une part non numérique EST déjà un bon libellé (مقدمة) ;
            # une part absente ne mérite aucun libellé (livre mono-volume)
            "label_ar": None if not raw else (f"الجزء {raw}" if numeric else raw),
        })
        previous_part = part
    volume_of_page[page["page_id"]] = volumes[-1]["volume_id"]

print(f"\n{len(volumes)} volumes\n")
print("volume_id  part_number  label_ar")
for v in volumes:
    print(f"{v['volume_id']:<10} {v['part_number']:<12} {v['label_ar']}")

print("\n-> part_number va de 6 à 10, volume_id de 1 à 5 : les deux sont nécessaires.")
print("metadata annonçait volume_count_observed =", meta["volume_count_observed"])

In [ ]:
# 3. sequence_num applicatif : rang dense 1..N dans l'ordre de lecture.
#
#    Pourquoi ne pas réutiliser page_id comme séquence ? Parce que page_id est
#    global au corpus (ici 5 527 207 et suivants) alors que l'app affiche
#    « page 12 sur 35 » et pagine avec LIMIT/OFFSET sur sequence_num.
#
#    Pourquoi garder page_id comme clé primaire plutôt que 1..N ? Parce que les
#    marque-pages et la progression de lecture de l'utilisateur pointent dessus.
#    Si une page était ajoutée en amont, une numérotation 1..N décalerait tout et
#    corromprait silencieusement chaque position sauvegardée.

for rank, page in enumerate(pages, start=1):
    page["_sequence_num"] = rank
    page["_volume_id"] = volume_of_page[page["page_id"]]

print("seq  page_id    volume  page imprimée")
for p in pages[:6] + pages[-3:]:
    print(f"{p['_sequence_num']:<4} {p['page_id']:<10} {p['_volume_id']:<7} {p['page_num']}")

print("\nLa pagination imprimée redémarre à chaque volume (115..118 puis 168..171) :")
print("c'est normal, ce sont les pages de la revue d'origine.")

## Étape 4 — Du `body` source aux trois colonnes de texte

Chaque page est stockée sous **trois** formes, jamais une seule :

| Colonne | Rôle | Pourquoi séparée |
|---|---|---|
| `body_html` | affichage | doit rester fidèle à la mise en forme |
| `body_plain` | sélection, copie, citation | l'utilisateur ne doit jamais copier des balises |
| `body_search` | recherche | normalisé (sans voyelles ni variantes), donc **illisible** |

La règle d'or : **on ne modifie jamais le texte de rendu pour faciliter la recherche.**
La normalisation vit dans sa propre colonne.

### Ce qu'on fait des balises

Recensement réel sur 60 livres (41 619 pages) : `span` 34 934, `td` 284, `tr` 102,
`th` 58, `br` 25, `table` 18, `img` 18. Et des liens `<a href="inr://man-3654">`
dans les livres de hadith.

| Source | Sortie | Raison |
|---|---|---|
| `<span data-type='title'>` seul dans son segment | `<h2 class="title">` | c'est un titre de section |
| le même en milieu de phrase | `<span class="title">` | c'est une emphase, pas un titre |
| `<br>`, `<hr>` | conservés | gérés par les deux clients |
| `<a href="inr://...">X</a>` | `X` | `inr://` n'est pas résolvable, un lien mort est pire que pas de lien |
| `<table>` | un `<p>` par ligne, cellules jointes par ` ǀ ` | aucun client ne gère les tables ; sans séparateur les cellules se collent |
| `<img src="data:...">` | `<span class="figure">` + ligne dans `assets` | voir étape 6 |
| tout le reste | balise supprimée, texte conservé | c'est exactement ce que font les deux clients |

### Le piège des guillemets

Les attributs sont écrits `data-type='title'` dans certains livres, `data-type="title"`
dans d'autres, et `id=toc-3` sans guillemets du tout. La parade est structurelle :
**on ne recopie jamais un attribut de la source**, on ne réémet que des attributs
qu'on a fabriqués, toujours en guillemets doubles.

In [ ]:
# Un scanner de balises tolérant : accepte <a b='c'>, <a b="c"> et <a b=c>
TAG_RE = re.compile(r"<\s*(/?)([a-zA-Z][\w-]*)([^>]*?)/?\s*>")
ATTR_RE = re.compile(r"([\w-]+)\s*=\s*(?:\"([^\"]*)\"|'([^']*)'|([^\s>]+))")


def parse_attrs(raw: str) -> dict:
    out = {}
    for m in ATTR_RE.finditer(raw):
        out[m.group(1).lower()] = m.group(2) or m.group(3) or m.group(4) or ""
    return out


def escape(text: str) -> str:
    """Le texte qu'on réinjecte dans du HTML doit être échappé."""
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def flatten_table(segment: str) -> str:
    """<table> -> un <p> par ligne. La balise fermante n'est pas garantie."""
    rows = []
    for row in re.split(r"<\s*tr[^>]*>", segment, flags=re.I)[1:]:
        cells = [
            strip_html(c).strip()
            for c in re.split(r"<\s*t[dh][^>]*>", row, flags=re.I)[1:]
        ]
        cells = [c for c in cells if c]
        if cells:
            rows.append("<p>" + escape(" ǀ ".join(cells)) + "</p>")
    return "".join(rows)


print(parse_attrs(" data-type='title' id=toc-3 "))
print(parse_attrs(' data-type="title" id="toc-9"'))
print(flatten_table("<table dir=rtl><tr><td>الخزانة</td><td>خزانة الأدب</td></tr>"))

In [ ]:
def convert_segment(segment: str, assets: list) -> str:
    """Un segment (= un paragraphe source) -> un bloc HTML."""
    if re.search(r"<\s*table", segment, re.I):
        return flatten_table(segment)

    out = []          # morceaux de HTML produits
    title_depth = 0   # profondeur d'imbrication dans un span de titre
    saw_title = False # ce segment contient-il un titre ?
    title_id = None
    pos = 0

    for m in TAG_RE.finditer(segment):
        out.append(escape(segment[pos:m.start()]))
        pos = m.end()
        closing, name, raw = m.group(1) == "/", m.group(2).lower(), m.group(3)
        attrs = parse_attrs(raw)

        if name == "span":
            if closing:
                if title_depth:
                    title_depth -= 1
                    if title_depth == 0:
                        out.append("\x00END\x00")
            elif attrs.get("data-type") == "title":
                title_depth += 1
                saw_title = True
                title_id = attrs.get("id") or title_id
                out.append("\x00START\x00")
            elif title_depth:
                title_depth += 1
        elif name in ("br", "hr") and not closing:
            out.append(f"<{name}>")
        elif name == "img" and not closing:
            out.append(register_image(attrs.get("src", ""), assets))
        # toute autre balise (a, i, b, s0...) : supprimée, texte conservé

    out.append(escape(segment[pos:]))
    html = "".join(out)

    # Le titre couvre-t-il tout le segment ? Alors c'est un titre de section.
    stripped = html.replace("\x00START\x00", "", 1).replace("\x00END\x00", "", 1)
    whole = saw_title and html.startswith("\x00START\x00") and html.rstrip().endswith("\x00END\x00")

    if whole:
        anchor = f' id="{escape(title_id)}"' if title_id else ""
        return f'<h2 class="title"{anchor}>{stripped.strip()}</h2>'

    html = html.replace("\x00START\x00", '<span class="title">').replace("\x00END\x00", "</span>")
    return f"<p>{html}</p>" if html.strip() else ""


def convert_body(body: str, assets: list) -> str:
    """`body` source -> `body_html`. Les paragraphes sont séparés par des \\r."""
    blocks = []
    for segment in body.split("\r"):
        if not segment.strip():
            continue  # un enchaînement de \r n'est qu'un séparateur
        block = convert_segment(segment, assets)
        if block:
            blocks.append(block)
    return "".join(blocks)

## Étape 5 — Les images encodées en base64

363 balises `<img>` dans 88 livres, en `image/png` **et** `image/jpg`. Elles sont
responsables de lignes JSONL de plusieurs centaines de kilo-octets.

**Aucun des deux clients n'affiche d'images aujourd'hui** (ni la liste de balises
autorisées d'Electron ni le parseur Flutter ne connaissent `img`). On les sort donc
du texte, on les catalogue dans la table `assets`, et on laisse un marqueur qui
préserve le lien page ↔ image — de quoi tout reconstruire plus tard sans relire 19 Go.

Ce livre n'en contient aucune : on le démontre sur une image synthétique.

In [ ]:
import base64
import hashlib

try:
    from PIL import Image
except ImportError:
    Image = None  # dimensions indisponibles, le reste fonctionne

DATA_URI_RE = re.compile(r"^data:(image/[a-zA-Z0-9.+-]+);base64,(.*)$", re.S)
MIME_EXT = {"image/png": "png", "image/jpeg": "jpg", "image/gif": "gif", "image/webp": "webp"}


def register_image(src: str, assets: list) -> str:
    """Décode, déduplique, enregistre — et renvoie le marqueur à insérer."""
    m = DATA_URI_RE.match(src.strip())
    if not m:
        return ""
    mime = m.group(1).lower()
    if mime == "image/jpg":
        mime = "image/jpeg"          # `image/jpg` n'est pas un type MIME valide
    try:
        blob = base64.b64decode(m.group(2), validate=False)
    except Exception:
        return ""

    digest = hashlib.sha256(blob).hexdigest()
    for asset in assets:            # déduplication intra-livre
        if asset["sha256"] == digest:
            return f'<span class="figure" data-asset="{digest[:8]}"></span>'

    width = height = None
    if Image is not None:
        try:
            with Image.open(io.BytesIO(blob)) as im:
                width, height = im.size   # lecture d'en-tête, pas de décodage complet
        except Exception:
            pass

    assets.append({
        "asset_id": len(assets) + 1,
        "file_path": f"assets/{digest[:16]}.{MIME_EXT.get(mime, 'bin')}",
        "mime_type": mime,
        "sha256": digest,
        "width": width,
        "height": height,
    })
    return f'<span class="figure" data-asset="{digest[:8]}"></span>'


# Démonstration : un PNG 1x1 transparent, inséré deux fois -> un seul asset
PNG_1PX = ("iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAAC0lEQVR4nGNgAAIAAAUAAY27m"
           "/MAAAAASUVORK5CYII=")
demo_assets = []
demo = convert_body(
    f"نص قبل الصورة\r<img src='data:image/png;base64,{PNG_1PX}'>\r"
    f"<img src=\"data:image/png;base64,{PNG_1PX}\">\rنص بعد الصورة",
    demo_assets,
)
print(demo)
print("\nassets (dédupliqués) :", json.dumps(demo_assets, ensure_ascii=False, indent=2))

## Étape 6 — Les trois colonnes, sur une vraie page

On assemble : `body_html` par conversion, `body_plain` par suppression des balises
et décodage des entités, `body_search` par normalisation du texte plat.

`body_search` est dérivé de `body_plain`, **jamais** de `body_html` : ainsi aucun
nom de balise ne peut se retrouver dans l'index de recherche.

### Ce que fait `normalize_ar`

Mode « large » : suppression des harakāt (voyelles) et du tatweel (allongement),
puis `أ إ آ ٱ → ا`, `ى → ي`, `ة → ه`. Objectif : qu'une recherche trouve un mot
quelle que soit son orthographe dans l'édition. On perd en précision, on gagne
en rappel — un choix assumé, versionné par `book_releases.fts_version`.

In [ ]:
assets = []
page = titled  # la page qui porte un titre

body_html = convert_body(page["body"], assets)
body_plain = decode_entities(strip_html(body_html))
body_search = normalize_ar(body_plain)

print("--- SOURCE (repr, les \\r sont les séparateurs de paragraphe) ---")
print(repr(page["body"][:260]), "\n")
print("--- body_html (affichage) ---")
print(body_html[:320], "\n")
print("--- body_plain (copie/citation) ---")
print(body_plain[:220], "\n")
print("--- body_search (recherche, illisible et c'est voulu) ---")
print(body_search[:220])

In [ ]:
# Effet de la normalisation, mot à mot
for mot in ("الإسلام", "الديانة", "مُحَمَّد", "عيسى", "الصــلاة"):
    print(f"{mot:12s} -> {normalize_ar(mot)}")

print("\n-> `الإسلام` et `الاسلام` deviennent la même chaîne : c'est le but.")
print("-> Limite connue : les chiffres arabes ne sont pas repliés,")
print("   donc `١٥٩` ne se trouve pas en tapant `159`.")

## Étape 7 — Les notes de bas de page

37,8 % des pages en ont. Le README du dataset annonce des marqueurs `(¬N)` :
c'est faux. Dans la réalité on trouve des chiffres arabes entre parenthèses `(١)`
**ou**, comme dans ce livre, des chiffres nus collés au mot qu'ils annotent
(`آربا سماج١التي`).

On ne cherche pas à réconcilier marqueur et note : ce serait de l'interprétation.
Les notes sont stockées telles quelles, en **texte brut**, parce que les deux
clients les affichent en texte nu (`reader_page_view.dart` et `reader.js` créent
un nœud texte, pas du HTML). Y mettre du HTML afficherait des balises à l'écran.

Leur version normalisée existe quand même — mais uniquement dans l'index de
recherche, car la table `pages` n'a pas de colonne `footnotes_search`.

In [ ]:
def clean_footnotes(raw: str | None) -> str | None:
    """Notes -> texte brut, paragraphes séparés par \\n. Jamais de HTML.

    Attention à l'ordre : `strip_html` **supprime** les `\\r` (c'est voulu, il
    produit du texte pour `body_plain` où les `\\r` sont déjà devenus des blocs).
    Il faut donc convertir `\\r` -> `\\n` AVANT de l'appeler, sinon tous les
    paragraphes de notes se retrouvent collés bout à bout.
    """
    if not raw or not raw.strip():
        return None
    text = decode_entities(strip_html(raw.replace("\r", "\n")))
    text = "\n".join(part.strip() for part in text.split("\n") if part.strip())
    return text or None


with_notes = next(p for p in pages if p["footnotes"] and "\r" in p["footnotes"])
cleaned = clean_footnotes(with_notes["footnotes"])

print("page_id", with_notes["page_id"], "|", with_notes["footnotes"].count("\r"), "séparateurs \\r")
print("\n--- notes nettoyées (une note par ligne) ---")
for line in cleaned.split("\n"):
    print(" ", line[:110])

print("\n--- si on avait oublié l'ordre : tout est collé ---")
print(" ", decode_entities(strip_html(with_notes["footnotes"]))[:150])

print("\n--- normalisées (pour l'index seulement) ---")
print(normalize_ar(cleaned)[:160])

## Étape 8 — Le sommaire hiérarchique

`toc.jsonl` donne des entrées à plat avec un `parent_id` qui pointe vers un
`title_id` du même fichier. Deux choses à calculer :

1. **`level`** : la profondeur, obtenue en remontant les parents. L'app s'en sert
   pour indenter le sommaire sans avoir à reconstruire l'arbre à chaque affichage.
2. **`sequence_num`** : l'ordre d'affichage. `shamela_title_id` fait l'affaire
   (vérifié sur 120 livres : trier dessus place toujours un parent avant ses enfants).

On conserve `title_id` comme `toc_id` et `parent_id` comme `parent_toc_id` — même
raison que pour `page_id` : la stabilité entre deux imports.

**Détection de cycle obligatoire.** Un `parent_id` qui boucle ferait tourner le
calcul de profondeur à l'infini. On le détecte avant toute écriture.

In [ ]:
toc = list(read_jsonl(os.path.join(BOOK_DIR, "toc.jsonl")))
toc.sort(key=lambda t: t["shamela_title_id"])
print("entrées de sommaire :", len(toc), "| manifest :", manifest["toc_count"])

parent_of = {t["title_id"]: t["parent_id"] for t in toc}
page_ids = {p["page_id"] for p in pages}


def depth(title_id: int) -> int:
    """Profondeur 1..N, avec garde anti-cycle."""
    level, seen, current = 1, {title_id}, parent_of.get(title_id)
    while current is not None:
        if current in seen:
            raise ValueError(f"cycle dans le sommaire sur title_id={title_id}")
        seen.add(current)
        level += 1
        current = parent_of.get(current)
    return level


# Contrôles avant écriture
orphans = [t for t in toc if t["page_id"] not in page_ids]
dangling = [t for t in toc if t["parent_id"] is not None and t["parent_id"] not in parent_of]
print("entrées pointant une page inexistante :", len(orphans))
print("parents inexistants                   :", len(dangling))

for rank, entry in enumerate(toc, start=1):
    entry["_level"] = depth(entry["title_id"])
    entry["_sequence_num"] = rank

print("\nniv  page_id    titre")
for t in toc[:12]:
    print(f"{t['_level']:<4} {t['page_id']:<10} {'  ' * (t['_level'] - 1)}{t['title_text'][:60]}")

## Étape 9 — Écrire le fichier SQLite

Le DDL vient de `tools/_common.py`, jamais recopié ici.

### Les PRAGMA de construction

```sql
PRAGMA page_size = 4096;    -- doit précéder le premier CREATE TABLE
PRAGMA journal_mode = OFF;  -- fichier neuf et jetable : aucun journal utile
PRAGMA synchronous = OFF;   -- en cas de crash on relance l'import, on ne répare pas
PRAGMA temp_store = MEMORY;
PRAGMA foreign_keys = ON;   -- validation gratuite de pages.volume_id et toc.page_id
```

`foreign_keys = ON` mérite un mot : il transforme une erreur de mapping en
exception immédiate plutôt qu'en base silencieusement incohérente. Il impose
l'ordre d'insertion **volumes → pages → toc**.

On s'écarte volontairement du `journal_mode = WAL` recommandé par la doc : le WAL
sert à la lecture concurrente pendant l'écriture, ce qui n'arrive jamais ici.

In [ ]:
if os.path.exists(OUT_DB):
    os.remove(OUT_DB)

con = sqlite3.connect(OUT_DB)
con.execute("PRAGMA page_size = 4096")
con.execute("PRAGMA journal_mode = OFF")
con.execute("PRAGMA synchronous = OFF")
con.execute("PRAGMA temp_store = MEMORY")
con.execute("PRAGMA foreign_keys = ON")
con.executescript(BOOK_SCHEMA)

print("tables créées :")
for (name,) in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
    print("  ", name)

In [ ]:
EDITION_ID = f"sh-{meta['book_id']}"

con.execute("BEGIN")

# --- volumes (d'abord : les pages y font référence) -----------------------
con.executemany(
    "INSERT INTO volumes (volume_id, part_number, label_ar, sequence_num) VALUES (?,?,?,?)",
    [(v["volume_id"], v["part_number"], v["label_ar"], v["sequence_num"]) for v in volumes],
)

# --- pages ---------------------------------------------------------------
assets = []
hasher = hashlib.sha256()   # empreinte du contenu, tous body_html concaténés
page_rows, fts_rows = [], []

for page in pages:
    html = convert_body(page["body"], assets)
    plain = decode_entities(strip_html(html))
    search = normalize_ar(plain)
    notes = clean_footnotes(page["footnotes"])
    hasher.update(html.encode("utf-8"))

    page_rows.append((
        page["page_id"],            # page_id = id source, stable entre imports
        page["shamela_page_id"],
        page["_volume_id"],
        page["page_num"],           # numéro imprimé, peut être NULL
        page["_sequence_num"],      # rang dense 1..N
        html, plain, search,
        notes,
        None,                       # hints : mort dans tout le corpus
        sha256_text(html),
    ))
    fts_rows.append((
        page["page_id"],            # rowid <- voir étape 10
        page["page_id"],
        search,
        normalize_ar(notes) if notes else "",
    ))

con.executemany(
    """INSERT INTO pages (page_id, shamela_page_id, volume_id, printed_page_num,
                          sequence_num, body_html, body_plain, body_search,
                          footnotes, hints, content_hash)
       VALUES (?,?,?,?,?,?,?,?,?,?,?)""",
    page_rows,
)
con.executemany(
    "INSERT INTO pages_fts (rowid, page_id, body_search, footnotes_search) VALUES (?,?,?,?)",
    fts_rows,
)

# --- bornes des volumes ---------------------------------------------------
con.execute(
    """UPDATE volumes SET
           first_page_id = (SELECT MIN(page_id) FROM pages WHERE volume_id = volumes.volume_id),
           last_page_id  = (SELECT MAX(page_id) FROM pages WHERE volume_id = volumes.volume_id)"""
)

# --- sommaire -------------------------------------------------------------
con.executemany(
    """INSERT INTO toc (toc_id, parent_toc_id, page_id, title_text,
                        title_normalized, level, sequence_num, shamela_title_id)
       VALUES (?,?,?,?,?,?,?,?)""",
    [(t["title_id"], t["parent_id"], t["page_id"], t["title_text"],
      normalize_ar(t["title_text"]), t["_level"], t["_sequence_num"],
      t["shamela_title_id"]) for t in toc],
)

# --- images ---------------------------------------------------------------
con.executemany(
    "INSERT INTO assets (asset_id, file_path, mime_type, sha256, width, height) VALUES (?,?,?,?,?,?)",
    [(a["asset_id"], a["file_path"], a["mime_type"], a["sha256"], a["width"], a["height"])
     for a in assets],
)

# --- carte d'identité du fichier -----------------------------------------
con.execute(
    """INSERT INTO book_info (edition_id, source_book_id, shamela_id, title_ar,
                              schema_version, content_version, page_count, toc_count,
                              created_at, content_hash)
       VALUES (?,?,?,?,?,?,?,?,?,?)""",
    (EDITION_ID, meta["book_id"], meta["shamela_id"], meta["title_ar"],
     SCHEMA_VERSION, 1, len(pages), len(toc),
     manifest["extracted_at"],   # date source : le même import redonne le même octet
     hasher.hexdigest()),
)

con.commit()
print(f"{len(pages)} pages, {len(toc)} entrées de sommaire, {len(volumes)} volumes, "
      f"{len(assets)} images")

## Étape 10 — Le piège FTS5 « contentless »

`pages_fts` est déclarée `content=''`, dite *contentless* : l'index stocke les
mots mais **pas** le texte, ce qui divise la taille par deux. Le prix à payer est
rarement documenté :

> Une table FTS5 contentless ne restitue **aucune** valeur de colonne, y compris
> les colonnes marquées `UNINDEXED`.

Donc `SELECT page_id FROM pages_fts WHERE pages_fts MATCH '...'` renvoie `NULL`.
Le seul lien exploitable vers la table `pages` est le **rowid** — et il faut donc
l'écrire explicitement à l'insertion :

```sql
INSERT INTO pages_fts (rowid, page_id, body_search, footnotes_search) VALUES (?,?,?,?)
--                     ^^^^^ = page_id
```

Sans ce `rowid`, SQLite en attribue un séquentiel et le lien avec `pages` est
simplement faux dès que `page_id` n'est pas `1..N` — ce qui est notre cas ici,
puisque les `page_id` valent 5 527 207 et suivants.

Et on termine par `optimize` (pas `rebuild`, impossible sur une table contentless).

In [ ]:
con.execute("INSERT INTO pages_fts(pages_fts) VALUES('optimize')")
con.commit()

terme = normalize_ar("الهندية")
print("recherche de :", terme)

print("\ncolonne page_id (piégée) :",
      con.execute("SELECT page_id FROM pages_fts WHERE pages_fts MATCH ? LIMIT 3", (terme,)).fetchall())

print("rowid (exploitable)      :",
      con.execute("SELECT rowid FROM pages_fts WHERE pages_fts MATCH ? LIMIT 3", (terme,)).fetchall())

print("\n--- jointure réelle, classée par pertinence ---")
for page_id, seq, printed, extrait in con.execute(
    """SELECT p.page_id, p.sequence_num, p.printed_page_num, substr(p.body_plain, 1, 60)
       FROM pages_fts f
       JOIN pages p ON p.page_id = f.rowid
       WHERE pages_fts MATCH ?
       ORDER BY rank
       LIMIT 5""",
    (terme,),
):
    print(f"  page {seq:>3} (imprimée {printed}) : {extrait}…")

print("\n--- recherche restreinte aux notes ---")
print(con.execute(
    "SELECT rowid FROM pages_fts WHERE pages_fts MATCH ? LIMIT 5",
    (f"footnotes_search : {normalize_ar('الهندوسية')}",)).fetchall())

## Étape 11 — Validation puis finalisation

Sur 8 589 livres, un livre malformé ne doit jamais coûter les 8 588 autres :
l'importeur valide, et en cas d'échec **saute le livre en le journalisant** au lieu
de tout arrêter. Un livre recalé est aussi retiré du catalogue, pour que celui-ci
n'annonce jamais un fichier qui n'existe pas.

Puis on prépare le fichier pour la distribution :

```sql
PRAGMA journal_mode = DELETE;   -- plus de -wal/-shm à traîner
PRAGMA optimize;
VACUUM;                         -- réordonne physiquement et compacte
PRAGMA integrity_check;
```

In [ ]:
def check(label: str, condition: bool, detail=""):
    print(f"  [{'ok ' if condition else 'ÉCHEC'}] {label} {detail}")
    return condition


n_pages = con.execute("SELECT COUNT(*) FROM pages").fetchone()[0]
n_toc = con.execute("SELECT COUNT(*) FROM toc").fetchone()[0]
n_fts = con.execute("SELECT COUNT(*) FROM pages_fts").fetchone()[0]
seq_min, seq_max, seq_distinct = con.execute(
    "SELECT MIN(sequence_num), MAX(sequence_num), COUNT(DISTINCT sequence_num) FROM pages"
).fetchone()
orphan_vol = con.execute("SELECT COUNT(*) FROM pages WHERE volume_id IS NULL").fetchone()[0]

ok = True
ok &= check("pages == manifest", n_pages == manifest["page_count"], f"({n_pages})")
ok &= check("sommaire == manifest", n_toc == manifest["toc_count"], f"({n_toc})")
ok &= check("index FTS == pages", n_fts == n_pages, f"({n_fts})")
ok &= check("sequence_num dense et unique",
            (seq_min, seq_max, seq_distinct) == (1, n_pages, n_pages))
ok &= check("aucune page sans volume", orphan_vol == 0)
ok &= check("contraintes de clés étrangères",
            not con.execute("PRAGMA foreign_key_check").fetchall())

con.commit()
con.execute("PRAGMA journal_mode = DELETE")
con.execute("PRAGMA optimize")
con.execute("VACUUM")
ok &= check("integrity_check",
            con.execute("PRAGMA integrity_check").fetchone()[0] == "ok")
con.close()

ok &= check("aucun fichier -wal / -shm résiduel",
            not any(os.path.exists(OUT_DB + s) for s in ("-wal", "-shm")))

print(f"\nvalidation globale : {'OK' if ok else 'ÉCHEC'}")
print(f"fichier : {os.path.getsize(OUT_DB):,} octets")
print(f"sha256  : {sha256_file(OUT_DB)}")
print(f"source  : {os.path.getsize(os.path.join(BOOK_DIR, 'pages.jsonl')):,} octets "
      f"-> ratio ×{os.path.getsize(OUT_DB) / os.path.getsize(os.path.join(BOOK_DIR, 'pages.jsonl')):.1f}")

## Étape 12 — Relire le fichier comme le fait l'application

Dernière vérification, la seule qui compte vraiment : exécuter les requêtes que
l'app émet réellement. Elles sont dans `lib/repositories/sqlite_book_repository.dart`
et `src/main/book-repository.js` — les deux clients posent exactement les mêmes.

Si ces requêtes rendent le bon résultat, le fichier est utilisable.

In [ ]:
con = sqlite3.connect(f"file:{OUT_DB}?mode=ro", uri=True)  # comme l'app : lecture seule
con.row_factory = sqlite3.Row

info = con.execute("SELECT * FROM book_info").fetchone()
print(f"{info['title_ar']}  ({info['page_count']} pages, {info['toc_count']} entrées)\n")

print("--- SELECT * FROM volumes ORDER BY sequence_num ---")
for v in con.execute("SELECT * FROM volumes ORDER BY sequence_num"):
    print(f"  vol {v['sequence_num']} | numéro imprimé {v['part_number']} | "
          f"{v['label_ar']} | pages {v['first_page_id']}..{v['last_page_id']}")

print("\n--- SELECT * FROM pages ORDER BY sequence_num LIMIT 3 OFFSET 4 ---")
for p in con.execute("SELECT * FROM pages ORDER BY sequence_num LIMIT 3 OFFSET 4"):
    print(f"  seq {p['sequence_num']} | vol {p['volume_id']} | imprimée {p['printed_page_num']}")
    print(f"     html  : {p['body_html'][:80]}…")
    print(f"     notes : {(p['footnotes'] or '(aucune)')[:70]}…")

print("\n--- SELECT * FROM toc ORDER BY sequence_num (arbre) ---")
for t in con.execute("SELECT * FROM toc ORDER BY sequence_num LIMIT 10"):
    print(f"  {'    ' * (t['level'] - 1)}{t['title_text'][:60]}  -> page {t['page_id']}")

con.close()

## Ce que le script d'import fait en plus

Ce notebook traite **un** livre en mémoire. `tools/import_shamela.py` en traite
120 ou 8 589, ce qui impose plusieurs choses que la clarté seule ne justifiait pas ici :

| Sujet | Notebook | Script |
|---|---|---|
| Lecture | `list(read_jsonl(...))` — tout en mémoire | deux passes en flux ; le plus gros livre fait 251 Mo et 124 569 pages |
| Écriture | `executemany` d'un coup | vidage tous les 500 enregistrements **ou** 16 Mo de texte |
| Intégrité source | comptages du manifest | + SHA-256 des trois fichiers recalculé pendant la première passe |
| Échec | exception | livre sauté, `.sqlite` partiel supprimé, motif journalisé, on continue |
| Parallélisme | aucun | `ProcessPoolExecutor`, les plus gros livres lancés en premier |
| Sélection | un livre codé en dur | `--books-per-category 3` (défaut) ou `--all` |
| Catalogue | non produit | `catalog.sqlite` : catégories, auteurs, œuvres, éditions, releases, `catalog_fts` |
| Rapport | affichage | `import-report.json` + `.csv`, un enregistrement par livre |

### Les décisions à retenir

1. **L'ordre de lecture est `page_id`**, pas `sequence_num` — ce dernier a des doublons.
2. **`page_id` reste l'identifiant source** : c'est ce qui fait survivre les
   marque-pages et la progression à une réimportation.
3. **`part_number` ≠ ordinal du volume** : ici les volumes sont numérotés 6 à 10.
4. **Trois colonnes de texte**, jamais une : on ne dégrade pas l'affichage pour la recherche.
5. **`rowid = page_id` dans `pages_fts`**, sinon l'index est injoignable aux pages.
6. **Le DDL vit dans `tools/_common.py`** : une seule source de vérité, aucune dérive possible.

### Pour aller plus loin

```powershell
# le même travail, sur 3 livres de chacune des 40 catégories
python tools/import_shamela.py --books-per-category 3

# un livre précis, pour reproduire un problème
python tools/import_shamela.py --book-ids 5925

# le corpus entier (~60 Go de sortie, prévoir le disque)
python tools/import_shamela.py --all --jobs 8 --compress
```